In [ ]:
from pyspoc import Calculator, Config

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
# Add the parent of the current script's parent to sys.path
project_root = Path('./').resolve().parents[1]

In [ ]:
new_mod_path = os.path.join(project_root,'summary_stats','feature_covariance_stats.py')

In [ ]:
my_config_yaml = f"""
Statistics:
    pyspoc.statistics.basic:
        Covariance:
            schemes:
                empirical:
                    estimator: EmpiricalCovariance
Reducers:
  {new_mod_path}:
    MaxCovPerFeature:
        schemes:
          max_min_median:
            stats: 
              - max
              - min
              - median
    EigenAMGMRatio:
      schemes:
        eigen_amgm:        
    EigenEntropy:
      schemes:
        eigen_entropy:
"""

my_config_name = "new_config"
cfg = Config.from_yaml(my_config_name, my_config_yaml)

In [ ]:
# key = main folder, value = sub folder, None if no need

data_files_dict = {}

data_files_dict['isotropic_gaussian_cloud'] = None
data_files_dict['gaussian_homoskedastic_blobs'] = 'k3_std1.0'

data_files_dict['randomCov_gaussian_cloud'] = None
data_files_dict['ellipsoid_homoskedastic_blobs'] = 'k3_std1.0'


In [ ]:
## PARSE THE ISO AND ANISO GAUSSIAN FOLDERS

filename_dict = {}

for dataset_type in data_files_dict.keys():

    n_sample = 1000
    n_feature = 10
    base_dir = os.path.join(project_root,'data','synthetic_data',dataset_type,f'N{n_sample}_P{n_feature}')

    if data_files_dict[dataset_type] is None:
        subfolder_path = base_dir
    else:
        subfolder_path = os.path.join(base_dir,data_files_dict[dataset_type])        

    meta_file = os.path.join(subfolder_path, "meta.csv")
    meta_df = pd.read_csv(meta_file)

    ## change this to take sample_num from meta_df:
    filename_dict[dataset_type] = {v.split('_')[0] : os.path.join(subfolder_path,v) for v in meta_df['filename']}

In [ ]:
flat = {f"{k}_{sk}": sv for k, v in filename_dict.items() for sk, sv in v.items()}

In [ ]:
res_dict = {}

for sample_file in flat.keys():

    print(sample_file)

    sample_X = np.load(flat[sample_file])

    calc = Calculator(sample_X)
    calc.compute(cfg)
    results = calc.results

    res_dict[sample_file] = results

In [ ]:
pd.concat(res_dict)

In [ ]:
# temp_arr = np.random.normal(size=(10,10))
# mask = ~np.eye(temp_arr.shape[0], dtype=bool)
# off_diag_matrix = np.where(mask, temp_arr, np.nan)

In [ ]:
# sample_file = 'isotropic_gaussian_cloud_sample0'
# sample_X = np.load(flat[sample_file])

# calc = Calculator(sample_X)
# calc.compute(cfg)
# results = calc.results.copy()

# --------------------

# sample_file = 'isotropic_gaussian_cloud_sample1'
# sample_X = np.load(flat[sample_file])

# calc = Calculator(sample_X)
# calc.compute(cfg)
# results = calc.results.copy()


In [ ]:
# ReducedStatistics:
#   pyspoc.rstatistics.pca:
#     PCAEigenVectors:
#       schemes:
#         first_two:
#           principal_vectors:
#             - 1
#             - 2